In [32]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import sequence

In [33]:
word_index = imdb.get_word_index()
rev_dict = {value:key for key, value in word_index.items()}

In [34]:
model = load_model('IMDB_Sentiment.h5')

In [35]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (32, 128)              │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,539 (5.07 MB)

 Trainable params: 1,329,537 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [36]:
import re

def preprocess_text(text, vocab_size=10000):
    words = re.sub(r"[^\w\s']", "", text.lower()).split()
    encoded = []
    for w in words:
        i = word_index.get(w)
        encoded.append(i + 3 if i is not None and i + 3 < vocab_size else 2)
    return sequence.pad_sequences([encoded], maxlen=500)

In [37]:
def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]



In [38]:
# Example review for prediction
example_review = "This movie was good. I really liked it"

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
Review: This movie was good. I really liked it
Sentiment: Positive
Prediction Score: 0.9138580560684204


In [39]:
tests = [
    "This movie was fantastic! The acting was great and the plot was thrilling.",
    "Absolutely terrible. Boring plot and awful acting, a waste of time.",
    "One of the best films I have seen, I loved every minute of it.",
]
for t in tests:
    print(predict_sentiment(t), '-', t)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
('Positive', np.float32(0.8694757)) - This movie was fantastic! The acting was great and the plot was thrilling.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
('Negative', np.float32(0.0012432981)) - Absolutely terrible. Boring plot and awful acting, a waste of time.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
('Positive', np.float32(0.9695154)) - One of the best films I have seen, I loved every minute of it.
